# Setup

In [1]:
import io
import requests
import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.experimental.io import check_path
from beak.experimental.conversions import create_binary_raster


In [2]:
# Set base paths and files
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 500
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "laculi_southwest"

BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")
base_raster = rasterio.open(BASE_RASTER)

# Points file and query to select relevant mineral occurences
PATH_LABELS = BASE_PATH / "RAW" / "mineral_deposits" / "Lacustrine_Lithium" / "TA2_Pre_Hack_12M" / "set_20240725"
check_path(PATH_LABELS)

# Set the output file
PATH_ROOT = BASE_PATH / "PROCESSED" / str("regional" + "_" + BASE_EXTENT + "_" + BASE_SPATIAL)
PATH_EXPORT = PATH_ROOT / "labels" / str("TA2_240725_FILTERED" + ".tif")
OUT_FILE = PATH_EXPORT

print(f"Output raster: {OUT_FILE}")

Output raster: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\PROCESSED\regional_102008_500_laculi_southwest\labels\TA2_240725_FILTERED.tif


In [3]:
token = ""


# Pull data from CDR

In [4]:
url_1 = 'https://api.cdr.land/v1/minerals/dedup-site/search/Lithium?with_location=true&with_deposit_types_only=true&system=minmod&top_n=1&limit=-1'
url_2 = 'https://api.cdr.land/v1/minerals/dedup-site/search/?with_location=true&with_deposit_types_only=true&system=minmod&top_n=1&limit=-1'

url = url_1

headers = {
    'accept': 'application/json',
    'Authorization': f'Bearer {token}'
}

response = requests.get(url, headers=headers)
data = response.text


In [5]:
file = io.StringIO(data)
df = pd.read_csv(file)
df


,id,record_ids,mineral_site_ids,names,type,rank,country,province,crs,centroid,...,contained_metal_unit,tonnage,tonnage_unit,grade,grade_unit,top1_deposit_type,top1_deposit_group,top1_deposit_environment,top1_deposit_classification_confidence,top1_deposit_classification_source
0,002ea762abc147288cd08c09a579c923,886a98dbf435439487b4ab7fd80306db,https://minmod.isi.edu/resource/site__mrdata-u...,Line Boy Mine,Past Producer,B,United States,Arizona,EPSG:4326,POINT (-110.6915 31.33542),...,million tonnes,NaN,million tonnes,NaN,percent,Breccia pipe copper,Breccia pipe,Magmatic hydrothermal,0.109823,"algorithm predictions, SRI deposit type classi..."
1,006f4d1a10ea4cf5ac17e86799782788,a976c33d322547bfb008a62e8cb717c2,https://minmod.isi.edu/resource/site__mrdata-u...,Anderson Mica Mines and Swanson Lithia Mine,Past Producer,B,United States,Connecticut,EPSG:4326,POINT (-72.52152 41.51499),...,million tonnes,NaN,million tonnes,NaN,percent,Simple pegmatite,Pegmatite,Magmatic,1.000000,"algorithm predictions, SRI crosswalk agent v0"
2,00f704d0e0484723bf0a1575b3251128,093f6bd12cb24c7996d9476655a109c4,https://minmod.isi.edu/resource/site__mrdata-u...,Carbonate Belle Claim,Occurrence,D,United States,Wyoming,EPSG:4326,POINT (-104.69134 41.30624),...,million tonnes,NaN,million tonnes,NaN,percent,Carbonatite niobium,Carbonatite,Magmatic,0.101381,"algorithm predictions, SRI deposit type classi..."
3,014ee2593aef46c9b3c68d35de28eab3,a86e483dcff744958707a0df18a34ec3,https://minmod.isi.edu/resource/site__mrdata-u...,Vanderburg - Katerina Pegmatite,Producer,D,United States,California,EPSG:4326,POINT (-117.08419 33.36672),...,million tonnes,NaN,million tonnes,NaN,percent,Simple pegmatite,Pegmatite,Magmatic,1.000000,"algorithm predictions, SRI crosswalk agent v0"
4,019dfd77586f47fa821d7fa9261e1959,fd874542c49344349b3ccdc0f5aa430b,https://minmod.isi.edu/resource/site__mrdata-u...,Unknown Name,Occurrence,D,Ecuador,Pichincha,EPSG:4326,POINT (-78.34978 -0.11507),...,million tonnes,NaN,million tonnes,NaN,percent,Lacustrine clay lithium,Clay,Basin evaporative,0.406181,"algorithm predictions, SRI deposit type classi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
452,fbb30fb681554201963deb19ea4c77b6,57943546ae014d26bc9611819ee38745,https://minmod.isi.edu/resource/site__mrdata-u...,Perry Prospect,Prospect,D,United States,Maine,EPSG:4326,POINT (-70.46082 44.52136),...,million tonnes,NaN,million tonnes,NaN,percent,LCT pegmatite,Pegmatite,Magmatic,0.305974,"algorithm predictions, SRI deposit type classi..."
453,fc6b7849d50242c8a721a711136c59e3,2ca981f598b048feae1355687b6ed061,https://minmod.isi.edu/resource/site__mrdata-u...,Mdara-Nigel Complex,Producer,D,Zimbabwe,NaN,EPSG:4326,POINT (31.43464 -19.9302),...,million tonnes,NaN,million tonnes,NaN,percent,Greisen beryllium±Li,Greisen,Magmatic hydrothermal,0.635058,"algorithm predictions, SRI deposit type classi..."
454,fc7ea10595744ddb96ed61c4849c07f4,7981f648d5d8449a8d71333fc2691ed1,https://minmod.isi.edu/resource/site__mrdata-u...,Stalin,Past Producer,D,United States,South Dakota,EPSG:4326,POINT (-103.62637 43.85496),...,million tonnes,NaN,million tonnes,NaN,percent,Peralkaline igneous HFSE- REE,Peralkaline igneous,Magmatic,0.378105,"algorithm predictions, SRI deposit type classi..."
455,fcff002c50ac4226971afcf238689e39,fd46fa8a38e84000b24e2b1922d0d8d3,https://minmod.isi.edu/resource/site__mrdata-u...,Dunton Mine,Past Producer,D,United States,Maine,EPSG:4326,POINT (-70.72363 44.54276),...,million tonnes,NaN,million tonnes,NaN,percent,Simple pegmatite,Pegmatite,Magmatic,0.388430,"algorithm predictions, SRI deposit type classi..."


## Create geodataframe and clear data

In [6]:
mineral_sites = df.copy()
mineral_sites["geometry"] = mineral_sites["wkt"].apply(loads)
mineral_sites = gpd.GeoDataFrame(mineral_sites, crs="EPSG:4326")
mineral_sites = mineral_sites.explode(column="geometry", ignore_index=True)
mineral_sites = mineral_sites.drop_duplicates(subset=["geometry"])
mineral_sites


,id,record_ids,mineral_site_ids,names,type,rank,country,province,crs,centroid,...,tonnage,tonnage_unit,grade,grade_unit,top1_deposit_type,top1_deposit_group,top1_deposit_environment,top1_deposit_classification_confidence,top1_deposit_classification_source,geometry
0,002ea762abc147288cd08c09a579c923,886a98dbf435439487b4ab7fd80306db,https://minmod.isi.edu/resource/site__mrdata-u...,Line Boy Mine,Past Producer,B,United States,Arizona,EPSG:4326,POINT (-110.6915 31.33542),...,NaN,million tonnes,NaN,percent,Breccia pipe copper,Breccia pipe,Magmatic hydrothermal,0.109823,"algorithm predictions, SRI deposit type classi...",POINT (-110.69150 31.33542)
1,006f4d1a10ea4cf5ac17e86799782788,a976c33d322547bfb008a62e8cb717c2,https://minmod.isi.edu/resource/site__mrdata-u...,Anderson Mica Mines and Swanson Lithia Mine,Past Producer,B,United States,Connecticut,EPSG:4326,POINT (-72.52152 41.51499),...,NaN,million tonnes,NaN,percent,Simple pegmatite,Pegmatite,Magmatic,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-72.52152 41.51499)
2,00f704d0e0484723bf0a1575b3251128,093f6bd12cb24c7996d9476655a109c4,https://minmod.isi.edu/resource/site__mrdata-u...,Carbonate Belle Claim,Occurrence,D,United States,Wyoming,EPSG:4326,POINT (-104.69134 41.30624),...,NaN,million tonnes,NaN,percent,Carbonatite niobium,Carbonatite,Magmatic,0.101381,"algorithm predictions, SRI deposit type classi...",POINT (-104.69134 41.30624)
3,014ee2593aef46c9b3c68d35de28eab3,a86e483dcff744958707a0df18a34ec3,https://minmod.isi.edu/resource/site__mrdata-u...,Vanderburg - Katerina Pegmatite,Producer,D,United States,California,EPSG:4326,POINT (-117.08419 33.36672),...,NaN,million tonnes,NaN,percent,Simple pegmatite,Pegmatite,Magmatic,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-117.08419 33.36672)
4,019dfd77586f47fa821d7fa9261e1959,fd874542c49344349b3ccdc0f5aa430b,https://minmod.isi.edu/resource/site__mrdata-u...,Unknown Name,Occurrence,D,Ecuador,Pichincha,EPSG:4326,POINT (-78.34978 -0.11507),...,NaN,million tonnes,NaN,percent,Lacustrine clay lithium,Clay,Basin evaporative,0.406181,"algorithm predictions, SRI deposit type classi...",POINT (-78.34978 -0.11507)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,fbb30fb681554201963deb19ea4c77b6,57943546ae014d26bc9611819ee38745,https://minmod.isi.edu/resource/site__mrdata-u...,Perry Prospect,Prospect,D,United States,Maine,EPSG:4326,POINT (-70.46082 44.52136),...,NaN,million tonnes,NaN,percent,LCT pegmatite,Pegmatite,Magmatic,0.305974,"algorithm predictions, SRI deposit type classi...",POINT (-70.46082 44.52136)
636,fc6b7849d50242c8a721a711136c59e3,2ca981f598b048feae1355687b6ed061,https://minmod.isi.edu/resource/site__mrdata-u...,Mdara-Nigel Complex,Producer,D,Zimbabwe,NaN,EPSG:4326,POINT (31.43464 -19.9302),...,NaN,million tonnes,NaN,percent,Greisen beryllium±Li,Greisen,Magmatic hydrothermal,0.635058,"algorithm predictions, SRI deposit type classi...",POINT (31.43464 -19.93020)
637,fc7ea10595744ddb96ed61c4849c07f4,7981f648d5d8449a8d71333fc2691ed1,https://minmod.isi.edu/resource/site__mrdata-u...,Stalin,Past Producer,D,United States,South Dakota,EPSG:4326,POINT (-103.62637 43.85496),...,NaN,million tonnes,NaN,percent,Peralkaline igneous HFSE- REE,Peralkaline igneous,Magmatic,0.378105,"algorithm predictions, SRI deposit type classi...",POINT (-103.62637 43.85496)
638,fcff002c50ac4226971afcf238689e39,fd46fa8a38e84000b24e2b1922d0d8d3,https://minmod.isi.edu/resource/site__mrdata-u...,Dunton Mine,Past Producer,D,United States,Maine,EPSG:4326,POINT (-70.72363 44.54276),...,NaN,million tonnes,NaN,percent,Simple pegmatite,Pegmatite,Magmatic,0.388430,"algorithm predictions, SRI deposit type classi...",POINT (-70.72363 44.54276)


Export

In [7]:
out_file_csv = PATH_LABELS / "mineral_sites_01_no_filter.csv"
out_file_gpkg = PATH_LABELS / "mineral_sites.gpkg"

mineral_sites.to_csv(out_file_csv, index=False)
mineral_sites.to_file(out_file_gpkg, driver="GPKG", layer="mineral_sites_01_no_filter_wkt")


# Filtering

## Basic filter

In [8]:
QUERY = "top1_deposit_classification_confidence >= 0.5 and type in ('Past Producer', 'Prospect', 'Producer') and rank in ('A', 'B', 'C') and country == 'United States'"

mineral_sites_filtered = mineral_sites.copy()
mineral_sites_filtered = mineral_sites_filtered.query(QUERY)
mineral_sites_filtered


,id,record_ids,mineral_site_ids,names,type,rank,country,province,crs,centroid,...,tonnage,tonnage_unit,grade,grade_unit,top1_deposit_type,top1_deposit_group,top1_deposit_environment,top1_deposit_classification_confidence,top1_deposit_classification_source,geometry
1,006f4d1a10ea4cf5ac17e86799782788,a976c33d322547bfb008a62e8cb717c2,https://minmod.isi.edu/resource/site__mrdata-u...,Anderson Mica Mines and Swanson Lithia Mine,Past Producer,B,United States,Connecticut,EPSG:4326,POINT (-72.52152 41.51499),...,NaN,million tonnes,NaN,percent,Simple pegmatite,Pegmatite,Magmatic,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-72.52152 41.51499)
7,02a6180f9b194903be554272862cc52c,"['01286193642b4b3b994f08697c9f2c23', '2ba57112...",['https://minmod.isi.edu/resource/site__mrdata...,['NI 43-101 Technical Report for the Kings Val...,Prospect,B,United States,Nevada,EPSG:4326,POINT (-118.0719325 41.722665),...,0.00009,million tonnes,0.27,percent,Lacustrine clay lithium,Clay,Basin evaporative,0.964834,"algorithm predictions, SRI deposit type classi...",POINT (-118.10909 41.76208)
8,02a6180f9b194903be554272862cc52c,"['01286193642b4b3b994f08697c9f2c23', '2ba57112...",['https://minmod.isi.edu/resource/site__mrdata...,['NI 43-101 Technical Report for the Kings Val...,Prospect,B,United States,Nevada,EPSG:4326,POINT (-118.0719325 41.722665),...,0.00009,million tonnes,0.27,percent,Lacustrine clay lithium,Clay,Basin evaporative,0.964834,"algorithm predictions, SRI deposit type classi...",POINT (-118.03477 41.68325)
25,081ba0867c3d4f88bad288f2d7830c8e,b7698b6a968a431189f9b876144e899f,https://minmod.isi.edu/resource/site__mrdata-u...,Mateen,Past Producer,B,United States,South Dakota,EPSG:4326,POINT (-103.57666 43.92306),...,NaN,million tonnes,NaN,percent,LCT pegmatite,Pegmatite,Magmatic,0.898854,"algorithm predictions, SRI deposit type classi...",POINT (-103.57666 43.92306)
44,0f9046fb2a5c4627a8518feb34d89c08,0be1bd61e8014623b12688b4adf9bb04,https://minmod.isi.edu/resource/site__mrdata-u...,White Basin,Past Producer,B,United States,Nevada,EPSG:4326,POINT (-114.57499 36.3311),...,NaN,million tonnes,NaN,percent,Lacustrine evaporite borate,Evaporite,Basin evaporative,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-114.57499 36.33110)
48,11a3113ef7004050955c85e7b84a813a,6ed547e6bc434a54890c59ec5068de38,https://minmod.isi.edu/resource/site__mrdata-u...,Rattlesnake Park Mine,Producer,B,United States,Colorado,EPSG:4326,POINT (-105.29837 40.35637),...,NaN,million tonnes,NaN,percent,Simple pegmatite,Pegmatite,Magmatic,0.576129,"algorithm predictions, SRI deposit type classi...",POINT (-105.29837 40.35637)
66,1a3bf6ebd7014c06a86b9c04f0d0c26f,3faba0ed896e4c9c9a146e4496198d74,https://minmod.isi.edu/resource/site__mrdata-u...,Kings Mountain Mine,Past Producer,C,United States,North Carolina,EPSG:4326,POINT (-81.34402 35.24821),...,NaN,million tonnes,NaN,percent,Greisen tin ±W-Mo,Greisen,Magmatic- Magmatic hydrothermal,0.555381,"algorithm predictions, SRI deposit type classi...",POINT (-81.34402 35.24821)
71,1cc0596a26174e3eac92142573c2c41d,3278fb7ba51748d8aee764d73c25a5d2,https://minmod.isi.edu/resource/site__mrdata-u...,Phelps Lode,Past Producer,B,United States,South Dakota,EPSG:4326,POINT (-103.61389 43.80802),...,NaN,million tonnes,NaN,percent,Simple pegmatite,Pegmatite,Magmatic,0.596119,"algorithm predictions, SRI deposit type classi...",POINT (-103.61389 43.80802)
81,1ffb757a71904c1db58ab5373356fd45,46a946b3e989473bb56c8147fccf244c,https://minmod.isi.edu/resource/site__mrdata-u...,Mount Wheeler,Past Producer,B,United States,Nevada,EPSG:4326,POINT (-114.34001 38.90106),...,NaN,million tonnes,NaN,percent,Vein tungsten,Vein,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-114.34001 38.90106)
106,26f241675e7d44f2a42ada89b6096707,60c51d8331674b46b00db17be93c80ba,https://minmod.isi.edu/resource/site__mrdata-u...,Century Lode Claim,Past Producer,B,United States,South Dakota,EPSG:4326,POINT (-103.57

Export

In [9]:
out_file_csv = PATH_LABELS / "mineral_sites_02_basic_filter.csv"
out_file_gpkg = PATH_LABELS / "mineral_sites.gpkg"

mineral_sites_filtered.to_csv(out_file_csv, index=False)
mineral_sites_filtered.to_file(out_file_gpkg, driver="GPKG", layer="mineral_sites_02_basic_filter_wkt")

In [10]:
column = "top1_deposit_type"
deposit_types = mineral_sites_filtered[column].unique()
deposit_types = pd.DataFrame(deposit_types, columns=[column])
deposit_types

,top1_deposit_type
0,Simple pegmatite
1,Lacustrine clay lithium
2,LCT pegmatite
3,Lacustrine evaporite borate
4,Greisen tin ±W-Mo
5,Vein tungsten
6,Epithermal beryllium
7,Lacustrine brine lithium
8,Epithermal mercury


## Final filtering

In [11]:
QUERY = "top1_deposit_type in ('Lacustrine clay lithium', 'Lacustrine evaporite borate', 'Lacustrine brine lithium') and top1_deposit_classification_confidence >= 0.5 and type in ('Past Producer', 'Prospect', 'Producer') and rank in ('A', 'B', 'C') and country == 'United States'"

mineral_sites_filtered = mineral_sites.copy()
mineral_sites_filtered = mineral_sites_filtered.query(QUERY)
mineral_sites_filtered

,id,record_ids,mineral_site_ids,names,type,rank,country,province,crs,centroid,...,tonnage,tonnage_unit,grade,grade_unit,top1_deposit_type,top1_deposit_group,top1_deposit_environment,top1_deposit_classification_confidence,top1_deposit_classification_source,geometry
7,02a6180f9b194903be554272862cc52c,"['01286193642b4b3b994f08697c9f2c23', '2ba57112...",['https://minmod.isi.edu/resource/site__mrdata...,['NI 43-101 Technical Report for the Kings Val...,Prospect,B,United States,Nevada,EPSG:4326,POINT (-118.0719325 41.722665),...,0.00009,million tonnes,0.27,percent,Lacustrine clay lithium,Clay,Basin evaporative,0.964834,"algorithm predictions, SRI deposit type classi...",POINT (-118.10909 41.76208)
8,02a6180f9b194903be554272862cc52c,"['01286193642b4b3b994f08697c9f2c23', '2ba57112...",['https://minmod.isi.edu/resource/site__mrdata...,['NI 43-101 Technical Report for the Kings Val...,Prospect,B,United States,Nevada,EPSG:4326,POINT (-118.0719325 41.722665),...,0.00009,million tonnes,0.27,percent,Lacustrine clay lithium,Clay,Basin evaporative,0.964834,"algorithm predictions, SRI deposit type classi...",POINT (-118.03477 41.68325)
44,0f9046fb2a5c4627a8518feb34d89c08,0be1bd61e8014623b12688b4adf9bb04,https://minmod.isi.edu/resource/site__mrdata-u...,White Basin,Past Producer,B,United States,Nevada,EPSG:4326,POINT (-114.57499 36.3311),...,NaN,million tonnes,NaN,percent,Lacustrine evaporite borate,Evaporite,Basin evaporative,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-114.57499 36.33110)
197,4b04f7aad3914bda8f750e58c4e9b476,c228de7de84a409c8795dd2212132cea,https://minmod.isi.edu/resource/site__mrdata-u...,Foote Lithium Brine Operation.,Producer,B,United States,Nevada,EPSG:4326,POINT (-117.5912 37.75855),...,NaN,million tonnes,NaN,percent,Lacustrine brine lithium,Brine,Basin evaporative,0.993826,"algorithm predictions, SRI deposit type classi...",POINT (-117.59120 37.75855)
228,5419818d0da74fba96f94b88f29cc4c5,0cc029ac236d404ab992d124e574db21,https://minmod.isi.edu/resource/site__mrdata-u...,Hector,Producer,B,United States,California,EPSG:4326,POINT (-116.45086 34.76668),...,NaN,million tonnes,NaN,percent,Lacustrine clay lithium,Clay,Basin evaporative,0.802968,"algorithm predictions, SRI deposit type classi...",POINT (-116.45086 34.76668)
510,cc4ba06d69f74e52809d8a325e4a75eb,b2e9a275619b426d955031ebb320b6ed,https://minmod.isi.edu/resource/site__mrdata-u...,Silver Peak Marsh,Producer,B,United States,Nevada,EPSG:4326,POINT (-117.63981 37.75272),...,NaN,million tonnes,NaN,percent,Lacustrine brine lithium,Brine,Basin evaporative,0.982076,"algorithm predictions, SRI deposit type classi...",POINT (-117.63981 37.75272)


Export

In [12]:
out_file_csv = PATH_LABELS / "mineral_sites_03_final_filter.csv"
out_file_gpkg = PATH_LABELS / "mineral_sites.gpkg"

mineral_sites_filtered.to_csv(out_file_csv, index=False)
mineral_sites_filtered.to_file(out_file_gpkg, driver="GPKG", layer="mineral_sites_03_final_filter_wkt")

# Create raster

In [13]:
data = mineral_sites_filtered.copy()
data = data.explode(ignore_index=True)
data = data[~data.is_empty]
data = data.reset_index(drop=True)
data = data.to_crs(base_raster.crs)


In [15]:
labels_array = create_binary_raster(data, base_raster, all_touched=False, same_shape=True, fill_negatives=True, out_file=OUT_FILE)
print(f"Number of positive training labels: {np.sum(labels_array==1)}")

Number of positive training labels: 6
